<a href="https://colab.research.google.com/github/mdaugherity/MachineLearning2026/blob/main/class/HW3_Trees_on_the_Titanic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from sklearn.datasets import fetch_openml
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split

# HW3 - Trees on the Titanic

Classify the people in data set and tell me **WHO LIVES AND WHO DIES**.

I given you lots of different example code in the past several tutorials. In this assignment you will have to pull the pieces together and demonstrate that you know what you are doing.  

The key skill in problem solving is the ability to break a complex problem into smaller steps.  In data science this often looks like making test cases that start as simply as possible and gradually add complexity to approach the original problem.  You should be able to confirm that each step works, or to put it another way:
**ALWAYS TEST YOUR CODE IN CASES WHERE YOU KNOW THE RIGHT ANSWER**







# SETUP - DONT CHANGE ANYTHING HERE

## Input
Load and process the input data.  

*Use the code below to load data without modifications.*

In [2]:
# Load Titanic
data = fetch_openml(name="titanic",version=1, as_frame=True, parser='auto')

df_raw = data.frame # the raw data
df_raw.head()

,pclass,survived,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked,boat,body,home.dest
0,1,1,"Allen, Miss. Elisabeth Walton",female,29.0000,0,0,24160,211.3375,B5,S,2,NaN,"St Louis, MO"
1,1,1,"Allison, Master. Hudson Trevor",male,0.9167,1,2,113781,151.5500,C22 C26,S,11,NaN,"Montreal, PQ / Chesterville, ON"
2,1,0,"Allison, Miss. Helen Loraine",female,2.0000,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"
3,1,0,"Allison, Mr. Hudson Joshua Creighton",male,30.0000,1,2,113781,151.5500,C22 C26,S,NaN,135.0,"Montreal, PQ / Chesterville, ON"
4,1,0,"Allison, Mrs. Hudson J C (Bessie Waldo Daniels)",female,25.0000,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"


Reminder from our **Titanic Pandas Tutorial** about some of these columns:

* pclass = Passenger Class 1, 2, or 3
* survived = 1 for people who survived, they will have a lifeboat number in boat and the body column should be blank.  Not all bodies are recovered.
* sibsp = number of siblings (for kids) or spouses (for adults) aboard
* parch = number of parents (for kids) or children (for adults) aboard
* fare is in old British money (pounds / shillings / pence) which gives weird fractions




In [3]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1309 entries, 0 to 1308
Data columns (total 14 columns):
 #   Column     Non-Null Count  Dtype   
---  ------     --------------  -----   
 0   pclass     1309 non-null   int64   
 1   survived   1309 non-null   category
 2   name       1309 non-null   object  
 3   sex        1309 non-null   category
 4   age        1046 non-null   float64 
 5   sibsp      1309 non-null   int64   
 6   parch      1309 non-null   int64   
 7   ticket     1309 non-null   object  
 8   fare       1308 non-null   float64 
 9   cabin      295 non-null    object  
 10  embarked   1307 non-null   category
 11  boat       486 non-null    object  
 12  body       121 non-null    float64 
 13  home.dest  745 non-null    object  
dtypes: category(3), float64(3), int64(3), object(5)
memory usage: 116.8+ KB


To use this for machine learning, we need to clean this up significantly:
* Throw away some columns
* Make all columns numeric (everything we use should be *int64* or *float64*)
* Remove rows with missing data


In [4]:
# Factorize any non-numeric columns we want to use
codes, genders = pd.factorize(df_raw.sex)
df_raw['gender'] = codes
print('Gender codes:', genders.categories)

Gender codes: Index(['female', 'male'], dtype='object')


In [5]:
print('Original data:')
print(df_raw.sex.value_counts())

print('\nFactorized result:')
df_raw.gender.value_counts()

Original data:
sex
male      843
female    466
Name: count, dtype: int64

Factorized result:


,count
gender,
1,843
0,466


In [6]:
# Survived is a category, so the values are strings instead of numbers!
df_raw.survived.unique()

['1', '0']
Categories (2, object): ['0', '1']

In [7]:
df_raw.survived = df_raw.survived.astype('int64')

In [8]:
df_raw.survived.unique()  # much better!

array([1, 0])

In [9]:
# Choose columns to keep
df_raw.columns

Index(['pclass', 'survived', 'name', 'sex', 'age', 'sibsp', 'parch', 'ticket',
       'fare', 'cabin', 'embarked', 'boat', 'body', 'home.dest', 'gender'],
      dtype='object')

In [10]:
features = ['pclass', 'gender', 'age', 'sibsp', 'parch', 'fare'] # choose columns for features
target = ['survived']
cols = target + features # combination of target and features
df = df_raw[cols].copy()
df.head()

,survived,pclass,gender,age,sibsp,parch,fare
0,1,1,0,29.0000,0,0,211.3375
1,1,1,1,0.9167,1,2,151.5500
2,0,1,0,2.0000,1,2,151.5500
3,0,1,1,30.0000,1,2,151.5500
4,0,1,0,25.0000,1,2,151.5500


In [11]:
df.describe()

,survived,pclass,gender,age,sibsp,parch,fare
count,1309.000000,1309.000000,1309.000000,1046.000000,1309.000000,1309.000000,1308.000000
mean,0.381971,2.294882,0.644003,29.881135,0.498854,0.385027,33.295479
std,0.486055,0.837836,0.478997,14.413500,1.041658,0.865560,51.758668
min,0.000000,1.000000,0.000000,0.166700,0.000000,0.000000,0.000000
25%,0.000000,2.000000,0.000000,21.000000,0.000000,0.000000,7.895800
50%,0.000000,3.000000,1.000000,28.000000,0.000000,0.000000,14.454200
75%,1.000000,3.000000,1.000000,39.000000,1.000000,0.000000,31.275000
max,1.000000,3.000000,1.000000,80.000000,8.000000,9.000000,512.329200


WARNING!  The count is different for different columns.  We must have some missing data!

In [12]:
# Clean up dataframe by dropping missing rows
print('Row count:\t', len(df))
df.dropna(inplace=True)  # delete rows with missing or bad values
print('After dropna:\t', len(df))

Row count:	 1309
After dropna:	 1045


In [13]:
# Save variables
X = df[features].values
y = df[target].values

In [14]:
print('Feature shape: ', X.shape)
print('Feature names: ', features)

Feature shape:  (1045, 6)
Feature names:  ['pclass', 'gender', 'age', 'sibsp', 'parch', 'fare']


## Load passenger data

In [15]:
df_pass = pd.read_csv('https://raw.githubusercontent.com/mdaugherity/MachineLearning2026/refs/heads/main/class/titanic_passengers.csv')
df_pass

,Name,pclass,gender,age,sibsp,parch,fare
0,Jackson,1,1,19,2,2,16
1,Jayden,1,1,91,1,1,74
2,Elliot,2,1,35,3,2,347
3,Barrett,3,1,-1,-1,8,412
4,Michael,1,1,37,1,3,359
5,Andrew,2,1,21,6,3,369
6,Levi,2,1,-101,1,2,324
7,Matthew,3,1,20,6,7,32
8,Musa,3,1,20,4,2,367
9,DrD,1,1,199,1,2,250


In [16]:
X_pred = df_pass[features].values # make sure we get the same feature columns in the same order
X_pred

array([[   1,    1,   19,    2,    2,   16],
       [   1,    1,   91,    1,    1,   74],
       [   2,    1,   35,    3,    2,  347],
       [   3,    1,   -1,   -1,    8,  412],
       [   1,    1,   37,    1,    3,  359],
       [   2,    1,   21,    6,    3,  369],
       [   2,    1, -101,    1,    2,  324],
       [   3,    1,   20,    6,    7,   32],
       [   3,    1,   20,    4,    2,  367],
       [   1,    1,  199,    1,    2,  250]])

# Problems - Your Work Goes Here
These problems are meant to be fairly short, so there are several of them...

## Problem 0 - Test Cases
Validate your code with these tests:
 * Make a new variable called ```X2``` with only age and fare columns.  
 * Train a tree using all rows of ```X2``` and ```y``` with max_depth=2
 * Print a nice diagram of the tree.  Make sure it is filled and all features and classes are labeled!
 * Make a plot of the decision boundaries and verify it matches the diagram


# Problem 1 - Tree Training
Using all 6 features, find the optimal value of max_depth and the test score for a DecisionTreeClassifier.

# Problem 2 - The Nearest Other Clasifier
Train a nearest neighbors classifier on the same data from problem 1 (all 6 features).  Clearly report the optimal parameters along with the train and test scores

# Problem 3 - Thinking
Is the decision tree or neighbors classifier better on this dataset?  **Explain and justify your answer.**

# Problem 4 - Predictions
Use the best classifier to classify the extra passengers.

# Problem 5 - Won't You Be My Neighbor
Determine which real Titanic passenger is most similar to "you" (i.e. your entry in the dataset) and print our their info.

# Problem 6 - The Important One
Finally, we see that age, passenger class, ticket fare, and gender are all important factors in determining survival.  **Which single factor is most important?**  Justify and explain your answer.

